In [1]:
pip install yfinance feedparser requests

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.8/137.8 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 96.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.1/510.1 kB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.6/215.6 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 14.5 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6090 sha256=5e62d4a35e6a20cc3106966dfefa34147b31f9d0e3bd812935111fc6be7950d2
  Stored in directory: /home/jovyan/.cache/pip/wheels/3b/25/2a/105d6a15df6914f4d15047691c6c28f9052cc1173e40285d03
Successfully built sgmlli

In [3]:
import yfinance as yf
import feedparser
import requests
import json
import time
from datetime import datetime


# 1. News Source: Yahoo Finance

def get_yahoo_news(ticker="NVDA"):
    print(f"Fetching Yahoo Finance news for {ticker}")
    stock = yf.Ticker(ticker)
    news_items = stock.news
    
    formatted_news = []
    for item in news_items:
        # Convert timestamp to readable date
        pub_date = datetime.fromtimestamp(item.get("providerPublishTime", 0)).strftime('%Y-%m-%d %H:%M:%S')
        
        formatted_news.append({
            "source_type": "News",
            "source_name": item.get("publisher", "Yahoo Finance"),
            "title": item.get("title", ""),
            "url": item.get("link", ""),
            "date": pub_date,
            "summary": item.get("relatedTickers", []) # Quick context on other mentioned companies
        })
    return formatted_news


# 2. Company Source: NVIDIA Official RSS

def get_nvidia_rss():
    print("Fetching NVIDIA official press releases")
    url = "https://nvidianews.nvidia.com/releases.xml"
    feed = feedparser.parse(url)
    
    formatted_rss = []
    for entry in feed.entries:
        formatted_rss.append({
            "source_type": "Company",
            "source_name": "NVIDIA Newsroom",
            "title": entry.title,
            "url": entry.link,
            "date": entry.published,
            "summary": entry.summary
        })
    return formatted_rss


# 3. Community Source: Hacker News API

def get_hn_discussions(query="nvidia", hitsPerPage=40):
    print("Fetching Hacker News discussions")
    url = f"http://hn.algolia.com/api/v1/search?query={query}&tags=story&hitsPerPage={hitsPerPage}"
    response = requests.get(url)
    
    formatted_hn = []
    if response.status_code == 200:
        data = response.json()
        for hit in data.get("hits", []):
            formatted_hn.append({
                "source_type": "Community",
                "source_name": "Hacker News",
                "title": hit.get("title", ""),
                "url": hit.get("url", hit.get("story_url", "")),
                "date": hit.get("created_at", ""),
                "summary": f"Points: {hit.get('points', 0)}, Comments: {hit.get('num_comments', 0)}"
            })
    return formatted_hn

# Main Execution Block

if __name__ == "__main__":
    all_documents = []
    
    # 1. Get news for NVIDIA and its core competitors
    tickers = ["NVDA", "AMD", "INTC", "TSM"]
    for t in tickers:
        all_documents.extend(get_yahoo_news(t))
        
    # 2. Get NVIDIA official news
    all_documents.extend(get_nvidia_rss())
    
    # 3. Get 100 Hacker News threads
    all_documents.extend(get_hn_discussions("nvidia", 100))
    
    # Save to JSON Lines format
    output_file = "nvidia_raw_intelligence.jsonl"
    
    with open(output_file, "w", encoding="utf-8") as f:
        for doc in all_documents:
            f.write(json.dumps(doc) + "\n")
            
    print(f"\nSuccessfully collected {len(all_documents)} documents.")

Fetching Yahoo Finance news for NVDA
Fetching Yahoo Finance news for AMD
Fetching Yahoo Finance news for INTC
Fetching Yahoo Finance news for TSM
Fetching NVIDIA official press releases
Fetching Hacker News discussions

Successfully collected 160 documents.
